<a href="https://colab.research.google.com/github/Ibrah-N/PNID_Custom_OCR_Training_Detector_Recognizer/blob/main/PINID_OCR_Classifier_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data

In [40]:
%cd /content/

/content


In [ ]:
!unzip /content/text_classification.zip

In [53]:
import os
import cv2
import shutil


def rotate_image_90(image):
  return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)


def copy_image(src_path, dst_path):
  shutil.copy(src_path, dst_path)


def write_image(image, image_path):
  cv2.imwrite(image_path, image)


def write_txt(txt_list, txt_path):

  with open(txt_path, "w") as f:
    for line in txt_list:
      f.write(line[0] + " " + line[1] + "\n")

  print("Text file written sucessfully to {}".format(txt_path))


def annotation_parse(image_root, save_root):
  """Parsing the images in image_root and saving it into images directory
  also listing the image_path and label
  """

  # processing images in rot_00
  files_list = []
  for idx, image_name in enumerate(os.listdir(image_root)):
    print("NO: {} - Processing Horizontal - Image: {}".format(idx, image_name))

    # copy image
    src_image_path = os.path.join(image_root, image_name)
    dst_image_path = os.path.join(save_root, image_name)

    copy_image(src_image_path, dst_image_path)

    files_list.append([os.path.join("images", image_name), "0"])

    if not src_image_path.lower().endswith(".jpg"):
      continue

    print("NO: {} - Processing Vertical - Image: {}".format(idx, image_name))

    # rotate image by 90 degree anti_clock_wise
    image = cv2.imread(src_image_path)
    image = rotate_image_90(image=image)

    if image is None:
      print(image, src_image_path)

    # save image
    dst_image_path = os.path.join(save_root, f"90_{image_name}")
    write_image(image, dst_image_path)

    files_list.append([os.path.join("images", f"90_{image_name}"), "1"])

  return files_list


In [54]:
image_root_train = "/content/text_classification/images_rot_00"
image_root_val = "/content/text_classification/val_rot_00"
save_root = "/content/text_classification_dataset/images"


# Dataset Structure
os.makedirs(save_root, exist_ok=True)
train_text_path = "/content/text_classification_dataset/train.txt"
val_text_path = "/content/text_classification_dataset/val.txt"



train_text = annotation_parse(image_root=image_root_train,
                 save_root=save_root)
write_txt(train_text, train_text_path)

val_text = annotation_parse(image_root=image_root_val,
                 save_root=save_root)
write_txt(val_text, val_text_path)

Streaming output truncated to the last 5000 lines.
NO: 794 - Processing Vertical - Image: image__2d487613-206a-45ca-9387-1a0aaf3a05af.jpg
NO: 795 - Processing Horizontal - Image: image__24b29cbb-491a-4fc9-9775-b7bb34d38e13.jpg
NO: 795 - Processing Vertical - Image: image__24b29cbb-491a-4fc9-9775-b7bb34d38e13.jpg
NO: 796 - Processing Horizontal - Image: image__3f117382-15c7-47cf-8f78-45616670511b.jpg
NO: 796 - Processing Vertical - Image: image__3f117382-15c7-47cf-8f78-45616670511b.jpg
NO: 797 - Processing Horizontal - Image: image__fc165d11-f964-491d-b8e7-30f18732f767.jpg
NO: 797 - Processing Vertical - Image: image__fc165d11-f964-491d-b8e7-30f18732f767.jpg
NO: 798 - Processing Horizontal - Image: image__9b6b54c6-9f38-4461-950a-b87d99a9e8f0.jpg
NO: 798 - Processing Vertical - Image: image__9b6b54c6-9f38-4461-950a-b87d99a9e8f0.jpg
NO: 799 - Processing Horizontal - Image: image__d02d06cd-6417-48f5-ac42-d402727b8321.jpg
NO: 799 - Processing Vertical - Image: image__d02d06cd-6417-48f5-ac42

# Classification Model Training

## Repo & Installation

In [1]:
!python3 -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu130/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 597.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.9/420.9 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!git clone https://github.com/PaddlePaddle/PaddleX.git

Cloning into 'PaddleX'...
remote: Enumerating objects: 177647, done.
remote: Counting objects: 100% (2036/2036), done.
remote: Compressing objects: 100% (725/725), done.
remote: Total 177647 (delta 1656), reused 1311 (delta 1311), pack-reused 175611 (from 3)
Receiving objects: 100% (177647/177647), 949.79 MiB | 27.87 MiB/s, done.
Resolving deltas: 100% (138041/138041), done.


In [3]:
!pip install ruamel.yaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.0 MB/s eta 0:00:00


In [4]:
!pip install ujson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 5.7 MB/s eta 0:00:00


In [1]:
!pip install paddleX

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.0/417.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 54.2 MB/s eta 0:00:00
  Attempting uninstall: PyYAML
    Found existing installation: PyYAML 6.0.3
    Uninstalling PyYAML-6.0.3:
      Successfully unins

In [3]:
!pip install langchain==1.2.15 langchain-community==0.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is inc

In [7]:
%cd /content/PaddleX
!pip install -e .

/content/PaddleX
Obtaining file:///content/PaddleX
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for paddlex (pyproject.toml) ... done
  Created wheel for paddlex: filename=paddlex-3.5.2-0.editable-py3-none-any.whl size=21374 sha256=33b4a9dcbf9fc2d348abd3283de4775aebf439436c7f2a9718a67b90286018d9
  Stored in directory: /tmp/pip-ephem-wheel-cache-nuwo6dkn/wheels/ef/07/03/5e0d55ec1d7bec8fad5d38007e86fab86c8c32c837dd98c74f
Successfully built paddlex
  Attempting uninstall: paddlex
    Found existing installation: paddlex 3.5.2
    Uninstalling paddlex-3.5.2:
      Successfully uninstalled paddlex-3.5.2


## Data

In [56]:
%cd /content/PaddleX

/content/PaddleX


In [29]:
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/data/textline_orientation_example_data.tar -P ./dataset
!tar -xf ./dataset/textline_orientation_example_data.tar -C ./dataset/

--2026-05-19 07:00:53--  https://paddle-model-ecology.bj.bcebos.com/paddlex/data/textline_orientation_example_data.tar
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9492480 (9.1M) [application/octet-stream]
Saving to: ‘./dataset/textline_orientation_example_data.tar’

textline_orientatio 100%[===================>]   9.05M  2.29MB/s    in 18s     

2026-05-19 07:01:13 (516 KB/s) - ‘./dataset/textline_orientation_example_data.tar’ saved [9492480/9492480]



In [57]:
!python main.py -c paddlex/configs/modules/textline_orientation/PP-LCNet_x0_25_textline_ori.yaml \
    -o Global.mode=check_dataset \
    -o Global.dataset_dir=./dataset/text_classification_dataset

Check dataset passed !


In [55]:
!rm -rf /content/PaddleX/dataset/text_classification_dataset

## Model Training

In [58]:
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-LCNet_x0_25_textline_ori_pretrained.pdparams

--2026-05-19 07:20:38--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-LCNet_x0_25_textline_ori_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:913:0:ff:b0a4:a156
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 994943 (972K) [application/octet-stream]
Saving to: ‘PP-LCNet_x0_25_textline_ori_pretrained.pdparams’

PP-LCNet_x0_25_text 100%[===================>] 971.62K  75.7KB/s    in 15s     

2026-05-19 07:20:55 (64.2 KB/s) - ‘PP-LCNet_x0_25_textline_ori_pretrained.pdparams’ saved [994943/994943]



In [61]:
%cd /content/PaddleX
!python main.py -c paddlex/configs/modules/textline_orientation/PP-LCNet_x0_25_textline_ori.yaml \
    -o Global.mode=train \
    -o Global.dataset_dir=./dataset/text_classification_dataset

/content/PaddleX
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
A new field (uniform_output_enabled) detected!
A new field (pdx_model_name) detected!
A new field (export_with_pir) detected!
['/usr/bin/python3', 'tools/train.py', '-c', '/root/.paddlex/tmpnj8n5ysa/clsmodel_PP-LCNet_x0_25_textline_ori.yml', '-o', 'Global.eval_during_train=True']

Log path: /content/PaddleX/output/train.log 

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2026/05/

In [60]:
!paddlex --install PaddleClas

Now download and update the repos: ['PaddleClas'].
Connecting to https://paddle-model-ecology.bj.bcebos.com/paddlex/PaddleX3.0/repos/PaddleClas.tar ...
[==================================================] 100.00%
Extracting PaddleClas.tar
[==================================================] 100.00%
remote: Total 0 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
From https://github.com/PaddlePaddle/PaddleClas
 * branch            develop    -> FETCH_HEAD
HEAD is now at e75732b update pre-commit (#3406)
All repos are existing.
Dependencies are listed below:
# PaddleClas dependencies
prettytable
ujson
opencv-contrib-python
pillow>=9.0.0
tqdm
PyYAML>=5.1
visualdl>=2.2.0
scipy>=1.0.0
scikit-learn>=0.21.0
gast==0.3.3
faiss-cpu
easydict
numpy==1.24.4; python_version<"3.12"
numpy==1.26.4; python_version>="3.12"
packaging
Now installing the packages...
Ignoring numpy: markers 'python_version < "3.12"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 

## Model Testing

In [63]:
!python main.py -c paddlex/configs/modules/textline_orientation/PP-LCNet_x0_25_textline_ori.yaml \
    -o Global.mode=evaluate \
    -o Global.dataset_dir=./dataset/text_classification_dataset

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
['/usr/bin/python3', 'tools/eval.py', '-c', '/root/.paddlex/tmpmydgm8ea/clsmodel_PP-LCNet_x0_25_textline_ori.yml']
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2026/05/19 07:43:57] ppcls INFO: 
==        PaddleClas is powered by PaddlePaddle !        ==
==                                                       ==
==   For more info please go to the following website.   ==
==                      

In [65]:
import wandb
from google.colab import userdata

wandb.login(key=str(userdata.get("wandb")))

wandb.init(
    project="PNID_OCR_Baidu",
    name = "Detection_Fine-Tuning_V1"
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ibrahnengineer to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [66]:
artifact = wandb.Artifact(
    name="Text_Classification_V1",
    type="model",
)

In [67]:
dir_path = "/content/PaddleX/output/best_model/inference"

artifact.add_dir(
    dir_path,
)

wandb.log_artifact(artifact)
wandb.finish()

wandb: Adding directory to artifact (/content/PaddleX/output/best_model/inference)... Done. 0.0s


In [ ]:
!pip uninstall -y numpy
!pip install numpy==1.26.4

In [2]:
from paddlex import create_model
model = create_model(model_name="PP-LCNet_x0_25_textline_ori",
                     model_dir="/content/PaddleX/output/best_model/inference")

ModuleNotFoundError: No module named 'langchain_text_splitters'

In [4]:
output = model.predict("/content/PaddleX/dataset/text_classification_dataset/images/90_image__0007a8f0-ddd7-4b90-83c9-90e3d67f30dd.jpg",  batch_size=1)
for res in output:
    res.print(json_format=False)
    res.save_to_img("./output_1/demo.png")
    res.save_to_json("./output_1/res.json")

{'res': {'input_path': '/content/PaddleX/dataset/text_classification_dataset/images/90_image__0007a8f0-ddd7-4b90-83c9-90e3d67f30dd.jpg', 'page_index': None, 'class_ids': array([[1]], dtype=int32), 'scores': array([0.99985], dtype=float32), 'label_names': ['180_degree']}}


In [6]:
!pip install pypdfium2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 77.3 MB/s eta 0:00:00


In [8]:
import os

ro = "/content/text_classification/val_rot_00"

for image in os.listdir(ro):

  sav_p = os.path.join(ro, image)

  output = model.predict(sav_p,  batch_size=1)
  for res in output:
      res.print(json_format=False)
      res.save_to_img(f"./output/{image}")

{'res': {'input_path': '/content/text_classification/val_rot_00/image__fb1779b9-ea5d-4ae8-9473-59c46982c770.jpg', 'page_index': None, 'class_ids': array([[0]], dtype=int32), 'scores': array([0.83233], dtype=float32), 'label_names': ['0_degree']}}
{'res': {'input_path': '/content/text_classification/val_rot_00/image__fee333ca-f81d-456e-bda2-712a8b23eff7.jpg', 'page_index': None, 'class_ids': array([[0]], dtype=int32), 'scores': array([0.99864], dtype=float32), 'label_names': ['0_degree']}}
{'res': {'input_path': '/content/text_classification/val_rot_00/image__fff8f9ba-8474-46d4-a1f3-25913fbca65c.jpg', 'page_index': None, 'class_ids': array([[0]], dtype=int32), 'scores': array([0.98629], dtype=float32), 'label_names': ['0_degree']}}
{'res': {'input_path': '/content/text_classification/val_rot_00/image__ffc6ac5d-7627-4e78-8aea-f7de99ab869e.jpg', 'page_index': None, 'class_ids': array([[0]], dtype=int32), 'scores': array([0.97358], dtype=float32), 'label_names': ['0_degree']}}
{'res': {'in

# Data Processing

## Load Text Classification Model

### Installations

In [ ]:
!python -m pip install paddlepaddle==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

In [ ]:
!python3 -m pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

In [2]:
!pip install paddleX

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.0/417.0 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 102.4 MB/s eta 0:00:00
  Attempting uninstall: PyYAML
    Found existing installation: PyYAML 6.0.3
    Uninstalling PyYAML-6.0.3:
      Successfully un

In [3]:
!pip install langchain==1.2.15 langchain-community==0.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is inc

In [4]:
!pip install pypdfium2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 56.0 MB/s eta 0:00:00


### Load Model

In [5]:
import wandb
from google.colab import userdata

wandb.login(key=str(userdata.get("wandb")))
api = wandb.Api()
artifact = api.artifact("ibrahnengineer/PNID_OCR_Baidu/Text_Classification_V1:v0")
artifact_dir = artifact.download()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ibrahnengineer to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb:   4 of 4 files downloaded.  


In [6]:
from paddlex import create_model
model = create_model(model_name="PP-LCNet_x0_25_textline_ori",
                     model_dir="/content/artifacts/Text_Classification_V1:v0")

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


In [ ]:
output = model.predict("/content/PaddleX/dataset/text_classification_dataset/images/90_image__0007a8f0-ddd7-4b90-83c9-90e3d67f30dd.jpg",  batch_size=1)
# for res in output:
#     res.print(json_format=False)
#     res.save_to_img("./output_1/demo.png")
#     res.save_to_json("./output_1/res.json")

## Processing Data

In [7]:
from google.colab import drive
drive.mount('/content/Ibrah')

Mounted at /content/Ibrah


### Load Data

In [ ]:
!unzip /content/Ibrah/MyDrive/OCR-Custom-Dataset/combine_images.zip

### Utils

In [30]:
import os
import cv2
import json


def predict_class(image):
  """Predicts the class of an text-image"""
  output = model.predict(image)

  class_ids, scores = 0, 0
  for res in output:
    class_ids = res.get("class_ids")
    scores = res.get("scores")


  return class_ids[0][0], scores[0]


def draw(image, bbox, text):


  color = (0, 255, 0)

  if str(text) == "1":
    color = (0, 0, 255)

  image = cv2.rectangle(image, (bbox[0], bbox[1]), (bbox[2], bbox[3]), color, 2)
  image = cv2.putText(image, str(text), (bbox[0], bbox[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

  return image


def crop_text(image, bbox):
  """Crop the image for the bounding box and return that cropped text image"""

  text_crop = image[bbox[1]:bbox[3], bbox[0]:bbox[2]]

  return text_crop



def get_box_list(bbox):
  """Convert boxes of dict into list of values"""
  return [int(bbox["x1"]), int(bbox["y1"]), int(bbox["x2"]), int(bbox["y2"])]


def save_image(image, image_path):
  cv2.imwrite(image_path, image)



def read_json(json_path):
  """read josn file"""

  if not json_path.lower().endswith(".json"):
    print("Invalid json path: {}".format(json_path))
    return None

  json_data = json.load(open(json_path))

  return json_data


def read_image(image_path):

  if not image_path.lower().endswith((".jpg", ".png")):
    print("Invalid image path: {}".format(image_path))
    return None

  # read image
  image = cv2.imread(image_path)

  if image is None:
    print("Image is None at image path: {}".format(image_path))
    return None

  return image


### Main

In [20]:
# == Paths ===
images_path = "/content/content/combine_images/images"
json_path = "/content/content/combine_images/combine_labels.json"

save_results_path = "/content/resutls"


os.makedirs(save_results_path, exist_ok=True)

In [31]:

# read json
json_data = read_json(json_path=json_path)

# parsing json_labels
for idx, x in enumerate(json_data):

  print("No: {} - Processing Image: {}".format(idx, x["ocr"]))

  ## extract metadata
  image_path = os.path.join(images_path, x["ocr"])
  bboxes = x['bbox']

  ## read image
  image = read_image(image_path=image_path)

  ## Paras the boxes
  for box in bboxes:

    ### get boxes list
    bbox = get_box_list(bbox=box)

    ### crop text-image
    crop_text_image = crop_text(image=image, bbox=bbox)

    ### detect_angle
    text_class, conf = predict_class(image=crop_text_image)

    ### draw over image
    drawn_image = draw(image=image, bbox=bbox, text=text_class)


  ## save image
  save_image(image=image, image_path=os.path.join(save_results_path, x["ocr"]))


  if idx==30:
    break

No: 0 - Processing Image: rot_00dg__4a46e8f4-images-0.jpg
No: 1 - Processing Image: rot_00dg__d9e8ebc0-images-1.jpg
No: 2 - Processing Image: rot_00dg__18c2726c-images-10.jpg
No: 3 - Processing Image: rot_00dg__8af00988-images-11.jpg
No: 4 - Processing Image: rot_00dg__d9d616d7-images-12.jpg
No: 5 - Processing Image: rot_00dg__65aa73fc-images-13.jpg
No: 6 - Processing Image: rot_00dg__c2d0a204-images-14.jpg
No: 7 - Processing Image: rot_00dg__dff48dad-images-15.jpg
No: 8 - Processing Image: rot_00dg__7cdb8b81-images-16.jpg
No: 9 - Processing Image: rot_00dg__789a991d-images-17.jpg
No: 10 - Processing Image: rot_00dg__c066412b-images-18.jpg
No: 11 - Processing Image: rot_00dg__482e63b5-images-19.jpg
No: 12 - Processing Image: rot_00dg__ca9cb0ce-images-2.jpg
No: 13 - Processing Image: rot_00dg__fc18454d-images-20.jpg
No: 14 - Processing Image: rot_00dg__74b5b7e1-images-21.jpg
No: 15 - Processing Image: rot_00dg__3726d647-images-22.jpg
No: 16 - Processing Image: rot_00dg__f87fb259-images-